# Extracting Edge Features for IPL Grid Cells

This notebook populates the existing cell-level CSV with edge-based image features.

The goal here is **not** to create train/test splits or train a model. The only goal is:

1. Read the existing feature CSV.
2. Find each referenced image in the Hugging Face dataset.
3. Compute edge maps for each image.
4. Extract edge features for each cell using `x_min`, `y_min`, `x_max`, and `y_max`.
5. Save a new populated CSV.

The output remains one row per `(image_file_name, cell_id)`.


## 1. Imports

These libraries are used for data handling, image loading, edge detection, and downloading the image dataset.


In [1]:
# Run this only if the libraries are not installed in your environment.
# %pip install pandas numpy opencv-python huggingface_hub

from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download


## 2. Configuration

Change only these values if your file names or output location changes.

`TARGET_CSV` is the CSV that already contains rows for image cells. This notebook will populate its edge-feature columns and write a new CSV.


In [2]:

# Hugging Face dataset containing the images.
HF_REPO_ID = "goyaljai/IPL-Player-Detection-IITB-PML"
HF_REPO_TYPE = "dataset"

# Annotations source CSV (wide format: one row per image, c01-c64 labels)
ANNOTATIONS_CSV = "Dataset_Annotations.csv"

# Output CSV
OUTPUT_CSV = "Individual_Feature_CSVs/Dataset_Features_Edge.csv"

# Project-required image size.
IMAGE_WIDTH = 800
IMAGE_HEIGHT = 600

# Project grid size.
GRID_ROWS = 8
GRID_COLS = 8

CELL_WIDTH = IMAGE_WIDTH // GRID_COLS      # 100 pixels
CELL_HEIGHT = IMAGE_HEIGHT // GRID_ROWS    # 75 pixels

# Edge feature columns to populate.
EDGE_FEATURE_COLUMNS = [
    "f_edge_density",
    "f_canny_count",
    "f_sobel_mean",
    "f_sobel_std",
    "f_laplacian_var",
    "f_contour_count",
]


## 3. Read the target CSV

This notebook assumes the CSV already has the proposed cell-level structure.

Important columns used for edge extraction:

- `image_file_name`: image to load
- `cell_id` / `cell_num`: cell identifier
- `x_min`, `y_min`, `x_max`, `y_max`: cell boundaries in the 800×600 image

The label columns such as `y_team_id` are not used to calculate edge features.


In [3]:

# Build cell-level DataFrame fresh from Dataset_Annotations.csv
annotations = pd.read_csv(ANNOTATIONS_CSV)

print("Loaded annotations:", annotations.shape)
print("Columns:", annotations.columns.tolist()[:8])

rows = []
for _, ann_row in annotations.iterrows():
    for cell_num in range(1, 65):
        cell_idx = cell_num - 1
        cell_row = cell_idx // GRID_COLS
        cell_col = cell_idx % GRID_COLS
        x_min = cell_col * CELL_WIDTH
        y_min = cell_row * CELL_HEIGHT
        x_max = x_min + CELL_WIDTH
        y_max = y_min + CELL_HEIGHT
        label_col = f"c{cell_num:02d}"
        rows.append({
            "Image File Name": ann_row["Image File Name"],
            "Train Or Test":   ann_row["Train Or Test"],
            "cell_row":        cell_row,
            "cell_col":        cell_col,
            "cell_num":        cell_num,
            "x_min":           x_min,
            "y_min":           y_min,
            "x_max":           x_max,
            "y_max":           y_max,
            "label":           int(ann_row[label_col]),
        })

df = pd.DataFrame(rows)
print(f"\nBuilt cell-level DataFrame: {len(df)} rows")
print("Columns:", df.columns.tolist())
df.head()


Loaded annotations: (1013, 67)
Columns: ['Image File Name', 'Train Or Test', 'count', 'c01', 'c02', 'c03', 'c04', 'c05']



Built cell-level DataFrame: 64832 rows
Columns: ['Image File Name', 'Train Or Test', 'cell_row', 'cell_col', 'cell_num', 'x_min', 'y_min', 'x_max', 'y_max', 'label']


,Image File Name,Train Or Test,cell_row,cell_col,cell_num,x_min,y_min,x_max,y_max,label
0,img_1.jpg,Train,0,0,1,0,0,100,75,0
1,img_1.jpg,Train,0,1,2,100,0,200,75,0
2,img_1.jpg,Train,0,2,3,200,0,300,75,0
3,img_1.jpg,Train,0,3,4,300,0,400,75,0
4,img_1.jpg,Train,0,4,5,400,0,500,75,0


## 4. Ensure required columns exist

The edge-feature columns are created if they are missing. Existing values are kept unless they are overwritten during feature extraction.

If cell boundary columns are missing but `cell_num` is present, the notebook can recreate the 8×8 grid boundaries.


In [4]:

# Add edge feature columns (initialized to NaN, will be filled in processing loop)
for col in EDGE_FEATURE_COLUMNS:
    df[col] = np.nan

print("Edge feature columns added.")
print("Current columns:", df.columns.tolist())
df.head()


Edge feature columns added.
Current columns: ['Image File Name', 'Train Or Test', 'cell_row', 'cell_col', 'cell_num', 'x_min', 'y_min', 'x_max', 'y_max', 'label', 'f_edge_density', 'f_canny_count', 'f_sobel_mean', 'f_sobel_std', 'f_laplacian_var', 'f_contour_count']


,Image File Name,Train Or Test,cell_row,cell_col,cell_num,x_min,y_min,x_max,y_max,label,f_edge_density,f_canny_count,f_sobel_mean,f_sobel_std,f_laplacian_var,f_contour_count
0,img_1.jpg,Train,0,0,1,0,0,100,75,0,NaN,NaN,NaN,NaN,NaN,NaN
1,img_1.jpg,Train,0,1,2,100,0,200,75,0,NaN,NaN,NaN,NaN,NaN,NaN
2,img_1.jpg,Train,0,2,3,200,0,300,75,0,NaN,NaN,NaN,NaN,NaN,NaN
3,img_1.jpg,Train,0,3,4,300,0,400,75,0,NaN,NaN,NaN,NaN,NaN,NaN
4,img_1.jpg,Train,0,4,5,400,0,500,75,0,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Download the image dataset and build an image index

This step downloads the Hugging Face dataset and builds a simple lookup table:

```text
image file name → full image path
```

We are **not** creating or analyzing train/test splits here. We are only locating the image files needed to populate edge features.

The dataset may internally store files in folders such as `train/` and `test/`, but this notebook treats all images as image files to be processed.


In [5]:
dataset_dir = Path(
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE
    )
)

print("Dataset directory:", dataset_dir)

# Build one image index across the entire dataset directory.
# This avoids split-specific logic because feature extraction only needs image files.
all_image_paths = (
    list(dataset_dir.rglob("*.jpg"))
    + list(dataset_dir.rglob("*.jpeg"))
    + list(dataset_dir.rglob("*.png"))
)

image_paths_by_name = defaultdict(list)
for path in all_image_paths:
    image_paths_by_name[path.name].append(path)

print("Total image files found:", len(all_image_paths))
print("Unique image file names found:", len(image_paths_by_name))

# Show duplicate file names, if any. Usually this should be empty.
duplicate_names = {name: paths for name, paths in image_paths_by_name.items() if len(paths) > 1}
print("Duplicate image file names:", len(duplicate_names))

if duplicate_names:
    first_name = next(iter(duplicate_names))
    print("Example duplicate name:", first_name)
    print(duplicate_names[first_name])


Fetching ... files: 0it [00:00, ?it/s]

Dataset directory: /Users/jai.goyal/.cache/huggingface/hub/datasets--goyaljai--IPL-Player-Detection-IITB-PML/snapshots/dcc5efca32ec25f03034661e69d41f782bf5698a
Total image files found: 1013
Unique image file names found: 1013
Duplicate image file names: 0


## 6. Helper function to locate an image

This function receives the image file name from the CSV and returns the path of the matching image.

If the same file name exists in more than one folder, it uses the first one and prints duplicate information in the previous cell. If duplicates become a real issue, we can add logic using `original_split` later, but that is not needed for simply creating edge features in most cases.


In [6]:
def find_image_path(image_file_name):
    """Return the local image path for a file name from the CSV."""

    matches = image_paths_by_name.get(str(image_file_name), [])

    if not matches:
        return None

    return matches[0]


## 7. Compute edge maps for one full image

For each image, edge maps are computed once on the full 800×600 image.

Then cell-wise features are extracted by cropping the full-image edge maps. This is better than running Canny/Sobel separately on each small cell because it avoids boundary artifacts and is more efficient.


In [7]:
def compute_edge_maps(image_bgr):
    """
    Compute edge-related maps for the full image.

    Input:
        image_bgr: OpenCV image in BGR format, already resized to 800x600.

    Output:
        Dictionary of full-image edge maps.
    """

    # Convert to grayscale because Canny, Sobel, and Laplacian are intensity-based.
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Reduce small noise before edge detection.
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Binary edge map. Non-zero pixels represent detected edges.
    canny = cv2.Canny(blurred, threshold1=50, threshold2=150)

    # Sobel gradients capture intensity changes in x and y directions.
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)

    # Gradient magnitude gives overall edge strength.
    sobel_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)

    # Laplacian captures high-frequency detail / sharpness.
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)

    return {
        "canny": canny,
        "sobel_mag": sobel_mag,
        "laplacian": laplacian,
    }


## 8. Extract edge features for one cell

Each row in the CSV represents one cell. This function crops the cell area from the full-image edge maps and calculates the required `f_` columns.


In [8]:
def extract_edge_features(edge_maps, x_min, y_min, x_max, y_max):
    """
    Extract edge features for a single cell.

    The coordinates come directly from the CSV columns:
    x_min, y_min, x_max, y_max.
    """

    # Crop this cell from each full-image edge map.
    canny_cell = edge_maps["canny"][y_min:y_max, x_min:x_max]
    sobel_cell = edge_maps["sobel_mag"][y_min:y_max, x_min:x_max]
    laplacian_cell = edge_maps["laplacian"][y_min:y_max, x_min:x_max]

    cell_area = canny_cell.shape[0] * canny_cell.shape[1]

    # Count non-zero Canny pixels. These are edge pixels.
    canny_count = int(np.count_nonzero(canny_cell))

    # Normalized edge density. For a 100x75 cell, area is 7500.
    edge_density = canny_count / cell_area if cell_area > 0 else 0.0

    # Average and variability of edge strength.
    sobel_mean = float(np.mean(sobel_cell))
    sobel_std = float(np.std(sobel_cell))

    # Variance of Laplacian: rough sharpness / high-frequency detail measure.
    laplacian_var = float(np.var(laplacian_cell))

    # Contours from Canny map. More contours often means more local structure.
    contours, _ = cv2.findContours(
        canny_cell,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    contour_count = int(len(contours))

    return {
        "f_edge_density": edge_density,
        "f_canny_count": canny_count,
        "f_sobel_mean": sobel_mean,
        "f_sobel_std": sobel_std,
        "f_laplacian_var": laplacian_var,
        "f_contour_count": contour_count,
    }


## 9. Populate the edge-feature columns

This is the main processing loop.

The loop groups the CSV by `image_file_name` so that each image is read and processed only once. For each image, it computes the full edge maps once, then fills the edge columns for every cell row belonging to that image.


In [9]:

from tqdm.auto import tqdm

missing_images = []
unreadable_images = []
processed_images = 0
processed_rows = 0

# Process one image at a time.
for image_file_name, group in tqdm(df.groupby("Image File Name", sort=False), desc="Processing images"):
    image_path = find_image_path(image_file_name)

    if image_path is None:
        missing_images.append(image_file_name)
        continue

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        unreadable_images.append(str(image_path))
        continue

    # Ensure the image is exactly 800x600, matching the project grid.
    image_bgr = cv2.resize(image_bgr, (IMAGE_WIDTH, IMAGE_HEIGHT))

    # Compute edge maps once for this image.
    edge_maps = compute_edge_maps(image_bgr)

    # Populate edge features for each cell row of this image.
    for idx in group.index:
        x_min = int(df.at[idx, "x_min"])
        y_min = int(df.at[idx, "y_min"])
        x_max = int(df.at[idx, "x_max"])
        y_max = int(df.at[idx, "y_max"])

        features = extract_edge_features(
            edge_maps=edge_maps,
            x_min=x_min,
            y_min=y_min,
            x_max=x_max,
            y_max=y_max,
        )

        for col, value in features.items():
            df.at[idx, col] = value

        processed_rows += 1

    processed_images += 1

print("Processed images:", processed_images)
print("Processed cell rows:", processed_rows)
print("Missing images:", len(missing_images))
print("Unreadable images:", len(unreadable_images))

if missing_images:
    print("First missing images:", missing_images[:10])

if unreadable_images:
    print("First unreadable image paths:", unreadable_images[:10])


Processing images:   0%|          | 0/1013 [00:00<?, ?it/s]

Processed images: 1013
Processed cell rows: 64832
Missing images: 0
Unreadable images: 0


## 10. Quick validation

This cell checks whether the edge columns were populated. If any values are still missing, they are likely from images that were not found or could not be read.


In [10]:

print("Missing values in edge columns:")
print(df[EDGE_FEATURE_COLUMNS].isna().sum())

print("\nPreview of populated edge columns:")
df[[
    "Image File Name",
    "Train Or Test",
    "cell_row",
    "cell_col",
    "label",
    *EDGE_FEATURE_COLUMNS,
]].head(20)


Missing values in edge columns:
f_edge_density     0
f_canny_count      0
f_sobel_mean       0
f_sobel_std        0
f_laplacian_var    0
f_contour_count    0
dtype: int64

Preview of populated edge columns:


,Image File Name,Train Or Test,cell_row,cell_col,label,f_edge_density,f_canny_count,f_sobel_mean,f_sobel_std,f_laplacian_var,f_contour_count
0,img_1.jpg,Train,0,0,0,0.060667,455.0,57.836068,40.741251,11.664933,12.0
1,img_1.jpg,Train,0,1,0,0.018933,142.0,37.648206,25.513181,5.905847,5.0
2,img_1.jpg,Train,0,2,0,0.040667,305.0,49.568082,37.760162,8.247790,12.0
3,img_1.jpg,Train,0,3,0,0.040400,303.0,39.715569,28.080953,6.231177,10.0
4,img_1.jpg,Train,0,4,0,0.016533,124.0,40.257663,27.651509,6.034347,4.0
5,img_1.jpg,Train,0,5,0,0.019867,149.0,36.037875,29.168826,5.672387,8.0
6,img_1.jpg,Train,0,6,0,0.031867,239.0,45.861416,31.002229,7.671646,7.0
7,img_1.jpg,Train,0,7,0,0.014133,106.0,37.181765,25.130286,5.802400,3.0
8,img_1.jpg,Train,1,0,0,0.075467,566.0,56.095497,44.671924,19.252837,18.0
9,img_1.jpg,Train,1,1,0,0.036000,270.0,44.724366,37.643692,14.066134,12.0


## 11. Save the populated CSV

The original CSV is not overwritten. The populated result is saved to `OUTPUT_CSV`.


In [11]:

# Drop helper columns not needed in final output
final_df = df.drop(columns=["cell_num", "x_min", "y_min", "x_max", "y_max"])

# Reorder: metadata first, then edge features
meta_cols = ["Image File Name", "Train Or Test", "cell_row", "cell_col", "label"]
feat_cols = EDGE_FEATURE_COLUMNS
final_df = final_df[meta_cols + feat_cols]

final_df.to_csv(OUTPUT_CSV, index=False)

print("Saved populated CSV to:", OUTPUT_CSV)
print("Final shape:", final_df.shape)
print("Columns:", final_df.columns.tolist())


Saved populated CSV to: Individual_Feature_CSVs/Dataset_Features_Edge.csv
Final shape: (64832, 11)
Columns: ['Image File Name', 'Train Or Test', 'cell_row', 'cell_col', 'label', 'f_edge_density', 'f_canny_count', 'f_sobel_mean', 'f_sobel_std', 'f_laplacian_var', 'f_contour_count']
